## Setting things up

### Import libraries
Add directory with the CIBUSmod modules to path to be able to import

In [1]:
import sys
import os
sys.path.insert(0, 'C:\\Users/jnka0003/Git repos/CIBUSmod')

Import CIBUSmod and packages for handling data and plotting

In [2]:
import CIBUSmod as cm

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


 -----------------------
|   Imported CIBUSmod   |
 -----------------------
commit : a382465f9d004487bd37148c40dccd05073d6472
branch : main
dirty  : True
remote : https://github.com/SLU-foodsystems/CIBUSmod
    


### Set up scenarios

In [3]:
# Create session
session = cm.Session(
    name = 'FORMAS',
    data_path = 'C:\\Users/jnka0003/Git repos/CIBUSmod/data',
    data_path_scenarios = 'scenarios',
    data_path_output = 'output',
)

# Define scenarios
session.add_scenario(
    name = 'BL',
    scenario_workbooks = 'BASELINE', 
    modules = 'all',
    pars = 'all',
    years = 0
)
session.add_scenario(
    name = 'MAX_CUR',
    scenario_workbooks = 'COMMON', 
    modules = 'all',
    pars = 'all',
    years = [70, 100, 110]
)
session.add_scenario(
    name = 'STEERS',
    scenario_workbooks = ['COMMON', 'STEERS'], 
    modules = 'all',
    pars = 'all',
    years = [70, 100, 110]
)
session.add_scenario(
    name = 'CUL_COWS',
    scenario_workbooks = ['COMMON', 'CUL_COWS'], 
    modules = 'all',
    pars = 'all',
    years = [70, 100, 110]
)
session.add_scenario(
    name = 'DRY_COWS',
    scenario_workbooks = ['COMMON', 'DRY_COWS'], 
    modules = 'all',
    pars = 'all',
    years = [70, 100, 110]
)
session.add_scenario(
    name = 'WIN_LAMB',
    scenario_workbooks = ['COMMON', 'WIN_LAMB'], 
    modules = 'all',
    pars = 'all',
    years = [70, 100, 110]
)
session.add_scenario(
    name = 'REC_HORSES',
    scenario_workbooks = ['COMMON', 'REC_HORSES'], 
    modules = 'all',
    pars = 'all',
    years = [70, 100, 110]
)
session.add_scenario(
    name = 'NAT_HORSES',
    scenario_workbooks = ['COMMON', 'NAT_HORSES'], 
    modules = 'all',
    pars = 'all',
    years = [70, 100, 110]
)
session.add_scenario(
    name = 'ALL',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES'], 
    modules = 'all',
    pars = 'all',
    years = [70, 100, 110]
)
session.add_scenario(
    name = 'ALL + NAT_HORSES',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'NAT_HORSES'], 
    modules = 'all',
    pars = 'all',
    years = [70, 100, 110]
)

A scenario with the name 'BL' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'MAX_CUR' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'STEERS' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'CUL_COWS' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'DRY_COWS' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'WIN_LAMB' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'REC_HORSES' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'NAT_HORSES' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + NAT_HORSES' already exists use .u

## Run scenarios (multi proc.)

In [4]:
# Import
from concurrent.futures import ProcessPoolExecutor, as_completed
import multi_proc as run

In [20]:
# Create list of scenarios to run
runs = [(s,y) for s,y in session.iterate('all')]

# scns = ['CUL_COWS','DRY_COWS','ALL','ALL + NAT_HORSES']
# runs = list(session.iterate(scns))

runs

[('BL', '0'),
 ('MAX_CUR', '70'),
 ('MAX_CUR', '100'),
 ('MAX_CUR', '110'),
 ('STEERS', '70'),
 ('STEERS', '100'),
 ('STEERS', '110'),
 ('CUL_COWS', '70'),
 ('CUL_COWS', '100'),
 ('CUL_COWS', '110'),
 ('DRY_COWS', '70'),
 ('DRY_COWS', '100'),
 ('DRY_COWS', '110'),
 ('WIN_LAMB', '70'),
 ('WIN_LAMB', '100'),
 ('WIN_LAMB', '110'),
 ('REC_HORSES', '70'),
 ('REC_HORSES', '100'),
 ('REC_HORSES', '110'),
 ('NAT_HORSES', '70'),
 ('NAT_HORSES', '100'),
 ('NAT_HORSES', '110'),
 ('ALL', '70'),
 ('ALL', '100'),
 ('ALL', '110'),
 ('ALL + NAT_HORSES', '70'),
 ('ALL + NAT_HORSES', '100'),
 ('ALL + NAT_HORSES', '110')]

In [21]:
%%time
# Do the multi-processing
with ProcessPoolExecutor(max_workers=8) as executor:
    
    futures = {executor.submit(run.do_run, session, scn_year) : scn_year for scn_year in runs}

    for future in as_completed(futures):
    
        scn, year = futures[future]
           
        try:
            t = future.result()
        except Exception as ee:
            print(f'(!!!) {scn}, {year} failed with the exception: {ee}')
        else:
            m = int(t/60)
            s = int(round(t - m*60))
            print(f'{scn}, {year} finished successfully in {m}min {s}s')
            
session.cache.clear()

BL, 0 finished successfully in 5min 34s
STEERS, 70 finished successfully in 9min 57s
MAX_CUR, 100 finished successfully in 10min 4s
STEERS, 100 finished successfully in 10min 29s
MAX_CUR, 70 finished successfully in 10min 35s
CUL_COWS, 70 finished successfully in 10min 47s
STEERS, 110 finished successfully in 11min 4s
MAX_CUR, 110 finished successfully in 11min 3s
CUL_COWS, 100 finished successfully in 6min 25s
CUL_COWS, 110 finished successfully in 9min 15s
DRY_COWS, 70 finished successfully in 10min 42s
DRY_COWS, 100 finished successfully in 10min 60s
DRY_COWS, 110 finished successfully in 11min 27s
WIN_LAMB, 110 finished successfully in 11min 5s
WIN_LAMB, 70 finished successfully in 11min 50s
WIN_LAMB, 100 finished successfully in 11min 40s
REC_HORSES, 70 finished successfully in 11min 2s
REC_HORSES, 100 finished successfully in 5min 16s
REC_HORSES, 110 finished successfully in 5min 17s
NAT_HORSES, 70 finished successfully in 5min 43s
NAT_HORSES, 100 finished successfully in 7min 42